Cross comparison of the AI model with other datasets


In [55]:
%pip install ultralytics kagglehub opencv-python

Note: you may need to restart the kernel to use updated packages.


In [56]:
from torch.utils.data import Dataset, Subset, DataLoader
import torch
import torchvision.transforms as T
from pathlib import Path
from io import BytesIO
from ultralytics import YOLO
import numpy as np
from functools import lru_cache
import kagglehub
import cv2
import logging
import concurrent.futures
import multiprocessing

In [ ]:
# Plant Village import
# path = kagglehub.dataset_download("mohitsingh1804/plantvillage", output_dir="datasets/plant_village", force_download=True)

# PlantDoc import
path = kagglehub.dataset_download(
    "andresmgs/plantdec", output_dir="datasets/plant_doc", force_download=True)

In [ ]:
# Model and basic function definition

potatoModel = YOLO("weights/potatoYOLOv11.pt")
tomatoModel = YOLO("weights/tomatoYOLOv11.pt")
onionModel = YOLO("weights/onionYOLOv11.pt")
lettuceModel = YOLO("weights/lettuceDisease.pt")
appleModel = YOLO("weights/appleDisease.pt")
strawberryModel = YOLO("weights/strawberryDisease.pt")


def predictBoxes(image, model):
    results = model(image)
    boxes = results[0].boxes
    output = []
    for box in boxes:
        x_min, y_min, x_max, y_max = box.xyxy[0].tolist()
        confidence = box.conf[0].item()
        class_id = int(box.cls[0].item())
        label = results[0].names[class_id]
        output.append({
            "label": label,
            "confidence": confidence,
            "box": [x_min, y_min, x_max, y_max]
        })
    return output


def get_label_tomato(image):
    result = predictBoxes(image, tomatoModel)
    return result


def get_label_potato(image):
    result = predictBoxes(image, potatoModel)
    return result


def get_label_onion(image):
    result = predictBoxes(image, onionModel)
    return result


def get_label_lettuce(image):
    result = predictBoxes(image, lettuceModel)
    return result


def get_label_apple(image):
    result = predictBoxes(image, appleModel)
    return result


def get_label_strawberry(image):
    result = predictBoxes(image, strawberryModel)
    return result


models = {
    "Potato": get_label_potato,
    "Tomato": get_label_tomato,
    "Onion": get_label_onion,
    "Lettuce": get_label_lettuce,
    "Apple": get_label_apple,
    "Strawberry": get_label_strawberry,
}

In [57]:
# adapted code for multiprocessing and batching
potatoModel = YOLO("weights/potatoYOLOv11.pt")
tomatoModel = YOLO("weights/tomatoYOLOv11.pt")
onionModel = YOLO("weights/onionYOLOv11.pt")
lettuceModel = YOLO("weights/lettuceDisease.pt")
appleModel = YOLO("weights/appleDisease.pt")
strawberryModel = YOLO("weights/strawberryDisease.pt")

models = {
    "Potato": potatoModel,
    "Tomato": tomatoModel,
    "Onion": onionModel,
    "Lettuce": lettuceModel,
    "Apple": appleModel,
    "Strawberry": strawberryModel,
}


def predictBoxes_batch(images, plant):
    results = models[plant](images)
    output = []
    for res in results:
        boxes = results[0].boxes
        output_per_image = []
        for box in boxes:
            x_min, y_min, x_max, y_max = box.xyxy[0].tolist()
            confidence = box.conf[0].item()
            class_id = int(box.cls[0].item())
            label = results[0].names[class_id]
            output_per_image.append({
                "label": label,
                "confidence": confidence,
                "box": [x_min, y_min, x_max, y_max]
            })
        output.append(output_per_image)

    return output

In [ ]:
# PlantDoc

# Dataset sub-datasets and Data Loader

# Full dataset containing all images from PlantDoc (including non detected diseases)

# Dataset to filter (Doesnt load images as it would be too much storage)

class PlantDoc_valid_labels_Dataset(Dataset):
    def __init__(self, dataset_path):
        self.ds_path: Path = Path(dataset_path)
        self.image_paths = self.get_all_paths()

    def __getitem__(self, idx):
        # Path chosen through indexing
        path_image: Path = self.image_paths[idx]
        # Creating the path for the labels
        path_parts = list(path_image.parts)
        path_parts[-2] = "labels"
        labels_path_wo_suffix = Path(*path_parts)
        path_labels = labels_path_wo_suffix.with_suffix(".txt")

        # Getting the labels from .txt file
        labels = []
        with open(path_labels, "r", encoding="utf-8") as file:
            for line in file:
                label, x1, y1, x2, y2 = map(str.strip, line.split(" "))
                labels.append({"label": label, "box": (x1, y1, x2, y2)})
        return (path_image, labels)

    def __len__(self):
        # Returns the length of the dataset
        return len(self.image_paths)

    def get_all_paths(self):
        # get all paths for the images in the dataset (labels paths are inferred later)
        image_paths = list(self.ds_path.rglob("**/*.jpg"))
        return image_paths


# Define which labels will be tested
plantDoc_labels = {"plants":
                   {
                       "Potato":
                       {
                           "labels": {"11": "EarlyBlight", "12": "LateBlight", "13": "Healthy"},
                           "idx": {"11", "12", "13"},
                       },
                       "Tomato":
                       {
                           "labels": {"19": "EarlyBlight", "20": "SeptoriaLeafSpot", "21": "BacterialSpot", "22": "TomatoLateBlight", "23": "MosaicVirus", "24": "YellowLeafCurlVirus", "25": "HealthyTomato", "26": "LeafMold", "27": "SpiderMites"},
                           # index
                           "idx": {"19", "20", "21", "22", "23", "24", "25", "26", "27"}
                       },
                   },
                   "used_idx": {"11", "12", "13", "19", "20", "21", "22", "23", "24", "25", "26", "27"}
                   }


# Checks if the label is valid
def is_valid(item, all_labels):
    _, item_labels = item
    for item_label in item_labels:
        if item_label["label"] not in all_labels["used_idx"]:
            return False
    if not item_labels:
        return False

    return True


# filters items
print("loading labels ds...")
labels_dataset = PlantDoc_valid_labels_Dataset("datasets/plant_doc")
print("labels ds finished. filtering items...")
filtered_items = [
    item for item in labels_dataset if is_valid(item, plantDoc_labels)]

print("filtering complete.")


# Filtered Dataset with labels
class PlantDoc_Dataset_distilled(Dataset):
    def __init__(self, items, labels_dict):
        self.resize_transform = T.Resize(size=(256, 256), antialias=True)
        self.max_labels = 16  # cuz i need uniform tensor sizes rip
        self.max_plants = 1
        self.items = items
        self.labels_dict = labels_dict

    def __getitem__(self, idx):
        img_path, original_labels = self.items[idx]
        labels = [item.copy() for item in original_labels]
        plants = set()
        for label in labels:
            for plant, dicts in self.labels_dict["plants"].items():
                if label["label"] in dicts["idx"]:
                    plants.add(plant)

        with open(img_path, "rb") as f:
            img_buffer = BytesIO(f.read())
        raw_bytes = np.frombuffer(img_buffer.getbuffer(), dtype=np.uint8)
        img_array_np = cv2.imdecode(raw_bytes, cv2.IMREAD_COLOR)[
            :, :, ::-1]  # reverts the colors into rgb
        img_tensor = torch.from_numpy(
            np.ascontiguousarray(img_array_np)).permute(2, 0, 1)
        img = self.resize_transform(img_tensor)

        plant_list = list(plants)
        plant_list.extend(["None",]*(self.max_plants-len(plant_list)))
        plant_list = plant_list[:self.max_plants]

        padding_dict = {"label": "None", "box": ("0", "0", "0", "0")}
        labels.extend([padding_dict] * (self.max_labels - len(labels)))
        labels = labels[:self.max_labels]

        return plant_list, img, labels

    def __len__(self):
        return len(self.items)


print("Creating new filtered dataset")
filtered_plantdoc_dataset = PlantDoc_Dataset_distilled(
    filtered_items, plantDoc_labels)

print("Creating data loader")
# loader = DataLoader(filtered_plantdoc_dataset, batch_size= 4, prefetch_factor=2, pin_memory=True, num_workers=2)
loader = DataLoader(filtered_plantdoc_dataset, batch_size=32,num_workers=4, prefetch_factor=3, persistent_workers=True)

print(" Getting batches")


def run(loader, models):
    for batch in loader:
        # create batches for each model
        plants, images, labels = batch
        plant_names = plants[0]
        unique_plants = set(plant_names)
        subbatches = dict()

        # Sorts batches per model into subbatches
        for plant in unique_plants:
            filter_mask = [name == plant for name in plant_names]
            plant_images = images[filter_mask]
            plant_labels = [labels for labels, bool_flag in zip(
                labels, filter_mask) if bool_flag]
            subbatches[plant] = [plant_images, plant_labels]
            print(subbatches[plant], "\n")
        print("New batch! \n subbatches for plants : \n ",
              "\n  ".join(subbatches.keys()))

        # Concurrency through thread pool executor (or process pool executor) for parallel processing
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = []
            for plant, subbatch in subbatches.items():
                imgs, correct_labels = subbatch
                future = executor.submit(
                    predictBoxes_batch, imgs, models[plant])
                futures.append[future]
            concurrent.futures.wait(futures)
            for ftr in futures:
                print(ftr)

            

splurt ur love juice in me daddy~
loading labels ds...
labels ds finished. filtering items...
filtering complete.
Creating new filtered dataset
Creating data loader
 Getting batches


Traceback (most recent call last):
  File "/home/willm277/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/multiprocessing/forkserver.py", line 344, in main
    code = _serve_one(child_r, fds,
                      unused_fds,
                      old_handlers)
  File "/home/willm277/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/multiprocessing/forkserver.py", line 384, in _serve_one
    code = spawn._main(child_r, parent_sentinel)
  File "/home/willm277/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'PlantDoc_Dataset_distilled'


BrokenPipeError: [Errno 32] Broken pipe